# Q6 — European Call Option Pricing

A risky asset price $S_t$ follows a geometric Brownian motion under the risk-neutral measure:

$$dS_t = r S_t\,dt + \sigma S_t\,d\beta_t, \quad S_0 = 10$$

with risk-free rate $r = 0.05$, volatility $\sigma = 0.03$.  
European call: strike $K = 10.05$, maturity $T = 1$.

---

## Part (a) — Black–Scholes Closed-Form Solution

Under $\mathbb{Q}$ the option value at $t=0$ is

$$C^E(0) = e^{-rT}\,\mathbb{E}^{\mathbb{Q}}[\max(S_T - K, 0)] = S_0\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2)$$

where

$$d_1 = \frac{\ln(S_0/K) + \left(r + \frac{\sigma^2}{2}\right)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

and $\Phi$ is the standard normal CDF.

**Interpretation of terms:**
- $S_0\,\Phi(d_1)$: present value of receiving the stock conditional on exercise
- $K e^{-rT}\,\Phi(d_2)$: present value of paying the strike; $\Phi(d_2) = \mathbb{Q}(S_T > K)$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Q6 parameters
S0    = 10.0
K     = 10.05
r     = 0.05
sigma = 0.03
T     = 1.0

print("Parameters loaded")

In [ ]:
def black_scholes_call(S0: float, K: float, r: float, sigma: float, T: float) -> float:
    """European call price via Black-Scholes closed form."""
    assert sigma > 0 and T > 0, "sigma and T must be positive"

    log_ratio = np.log(S0 / K)                  # log moneyness
    drift_adj = (r + 0.5 * sigma**2) * T        # risk-neutral drift + Ito correction
    denom     = sigma * np.sqrt(T)               # total volatility over [0, T]

    d1 = (log_ratio + drift_adj) / denom
    d2 = d1 - denom

    return S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


bs_price = black_scholes_call(S0, K, r, sigma, T)
print(f"Black-Scholes call price: C^E(0) = {bs_price:.6f}")

### Intermediate values (verification)

Expected: $d_1 \approx 1.515$, $d_2 \approx 1.485$, $C^E(0) \approx 0.4913$.

In [ ]:
log_ratio = np.log(S0 / K)
drift_adj = (r + 0.5 * sigma**2) * T
denom     = sigma * np.sqrt(T)
d1 = (log_ratio + drift_adj) / denom
d2 = d1 - denom

print(f"ln(S0/K)   = {log_ratio:.6f}")
print(f"drift_adj  = {drift_adj:.6f}")
print(f"denom      = {denom:.6f}")
print(f"d1         = {d1:.4f}")
print(f"d2         = {d2:.4f}")
print(f"Phi(d1)    = {norm.cdf(d1):.4f}")
print(f"Phi(d2)    = {norm.cdf(d2):.4f}")
print(f"C^E(0)     = {bs_price:.4f}")

---

## Part (b) — Monte Carlo via Euler–Maruyama

When no closed form is available the risk-neutral price is estimated via the Monte Carlo estimator:

$$\hat{V} = e^{-rT} \cdot \frac{1}{n_{\text{sim}}}\sum_{i=1}^{n_{\text{sim}}} \max\!\left(S_T^{(i)} - K,\; 0\right) \xrightarrow{\text{a.s.}} C^E(0) \quad \text{as } n_{\text{sim}} \to \infty$$

by the Strong Law of Large Numbers, since each $\max(S_T^{(i)}-K,0)$ is iid with finite expectation.

### Euler–Maruyama discretisation

On a uniform grid $t_0 = 0 < t_1 < \cdots < t_N = T$ with step $\Delta t = T/N$, the vectorised update is:

$$\mathbf{S} \leftarrow \mathbf{S} \cdot \left(1 + r\,\Delta t + \sigma\,\sqrt{\Delta t}\;\mathbf{Z}\right), \quad \mathbf{Z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}_{n_{\text{sim}}})$$

Only the current price vector $\mathbf{S} \in \mathbb{R}^{n_{\text{sim}}}$ is stored — not the full $N \times n_{\text{sim}}$ path matrix.

In [ ]:
def simulate_gbm_em(S0: float, r: float, sigma: float,
                    T: float, N: int, n_sim: int) -> np.ndarray:
    """Simulate n_sim GBM terminal prices via Euler-Maruyama."""
    dt    = T / N
    sq_dt = np.sqrt(dt)
    S     = np.full(n_sim, float(S0))
    for _ in range(N):
        Z = np.random.standard_normal(n_sim)
        S = S * (1.0 + r * dt + sigma * sq_dt * Z)
    return S

In [ ]:
def monte_carlo_call(S0: float, K: float, r: float, sigma: float,
                     T: float, N: int, n_sim: int) -> float:
    """Single Monte Carlo estimate of the European call price."""
    S_T     = simulate_gbm_em(S0, r, sigma, T, N, n_sim)
    payoffs = np.maximum(S_T - K, 0.0)
    return np.exp(-r * T) * payoffs.mean()

In [ ]:
def repeat_mc(S0: float, K: float, r: float, sigma: float,
              T: float, N: int, n_sim: int, n_repeats: int = 100) -> np.ndarray:
    """Run monte_carlo_call independently n_repeats times."""
    estimates = np.empty(n_repeats)
    for j in range(n_repeats):
        estimates[j] = monte_carlo_call(S0, K, r, sigma, T, N, n_sim)
    return estimates

In [ ]:
def plot_mc_histogram(estimates: np.ndarray, bs_price: float) -> None:
    """Histogram of MC estimates with Black-Scholes reference line."""
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.hist(estimates, bins=20, color="steelblue", edgecolor="white", alpha=0.85,
            label=f"MC estimates (n={len(estimates)})")
    ax.axvline(bs_price, color="crimson", linewidth=2, linestyle="--",
               label=f"Black–Scholes: {bs_price:.4f}")

    ax.set_xlabel("Monte Carlo call price estimate", fontsize=12)
    ax.set_ylabel("Frequency", fontsize=12)
    ax.set_title("Distribution of MC estimates vs Black–Scholes price (100 runs)", fontsize=13)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
N         = 1000
n_sim     = 10_000
n_repeats = 100

mc_estimates = repeat_mc(S0, K, r, sigma, T, N, n_sim, n_repeats)

print(f"MC mean  : {mc_estimates.mean():.6f}")
print(f"MC std   : {mc_estimates.std():.6f}")
print(f"BS price : {bs_price:.6f}")

plot_mc_histogram(mc_estimates, bs_price)

### Discussion

The histogram forms a roughly normal distribution (CLT applied to the sample mean) centred near the Black–Scholes reference, confirming SLLN convergence. The spread reflects the Monte Carlo standard error $\approx \sigma_P / \sqrt{n_{\text{sim}}}$ where $\sigma_P = \mathrm{std}(\max(S_T - K, 0))$; it shrinks as $n_{\text{sim}} \to \infty$.